# 04 — Fair KnottedGraph vs Topoly Yamada comparison

This notebook runs the validated identical-PD benchmarks. It does **not** use the misleading Topoly `coordinates + bridges -> yamada()` path.

Correctness is checked first. Topoly and KnottedGraph must agree up to the allowed Laurent unit \(\pm A^k\); otherwise timing is rejected.

Two suites are run:
1. decomposable diagrams, exposing scaling with crossing number;
2. connected spatial graphs, preventing the block-factorization optimization from being the only reason for good performance.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys, csv
import matplotlib.pyplot as plt

ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/"pyproject.toml").exists():
    ROOT=ROOT.parent
SRC=ROOT/"src"; sys.path.insert(0,str(SRC))
branch=subprocess.check_output(["git","rev-parse","--abbrev-ref","HEAD"],cwd=ROOT,text=True).strip()
if branch!="perf/yamada-max-optimization":
    raise RuntimeError(f"Expected perf/yamada-max-optimization, got {branch}")
import knotted_graph
assert SRC in Path(knotted_graph.__file__).resolve().parents
try:
    import topoly
except ImportError:
    raise ImportError("Install Topoly in this environment before running: pip install topoly")

OUT=ROOT/"User_guide"/"benchmarks"
RES=OUT/"results_latest"; FIG=OUT/"figures_latest"
RES.mkdir(exist_ok=True); FIG.mkdir(exist_ok=True)

In [ ]:
def run_summary(script,timeout=1800):
    env=dict(os.environ); env["PYTHONPATH"]=str(SRC)
    p=subprocess.run([sys.executable,str(ROOT/"dev"/script)],cwd=ROOT,env=env,
                     text=True,capture_output=True,timeout=timeout)
    print(p.stdout)
    if p.returncode:
        print(p.stderr)
        raise RuntimeError(f"{script} failed")
    for line in p.stdout.splitlines():
        if line.startswith("SUMMARY="):
            return json.loads(line[8:])
    raise RuntimeError(f"{script} did not emit SUMMARY=")

decomp=run_summary("benchmark_topoly_identical_pd.py")
connected=run_summary("benchmark_topoly_connected_pd.py")

In [ ]:
def save(name,rows):
    keys=list(dict.fromkeys(k for r in rows for k in r))
    with (RES/name).open("w",newline="") as f:
        w=csv.DictWriter(f,fieldnames=keys); w.writeheader(); w.writerows(rows)

save("04_topoly_identical_pd.csv",decomp)
save("04_topoly_connected_pd.csv",connected)

plt.figure(figsize=(8.5,5.3))
q=sorted(decomp,key=lambda r:r["crossings"])
plt.plot([r["crossings"] for r in q],[r["knottedgraph_s"] for r in q],marker="o",label="KnottedGraph")
plt.plot([r["crossings"] for r in q],[r["topoly_s"] for r in q],marker="o",label="Topoly")
plt.yscale("log"); plt.xlabel("Crossings c"); plt.ylabel("Yamada runtime (s)")
plt.title("Identical-PD Yamada benchmark"); plt.grid(alpha=.25); plt.legend(); plt.tight_layout()
plt.savefig(FIG/"04_topoly_identical_pd.pdf",bbox_inches="tight")
plt.savefig(FIG/"04_topoly_identical_pd.png",dpi=300,bbox_inches="tight")
plt.show()

In [ ]:
print("Connected-graph results:")
for r in connected:
    print(r)
print("\nAll timings above are accepted only after polynomial equivalence checks in the benchmark scripts.")